# LLM 7일차 (1) — LangGraph 조건부 엣지 : `add_conditional_edges` · 라우팅 함수 · 경로 매핑 · 반복과 `recursion_limit`

교재 `AI Agent` **p.96~99** (조건부 엣지) 를 실행해 확인한 노트입니다. 검색이 필요한 질문만 Tavily 로 보내는 실습은 [`langgraph_search_router_agent.ipynb`](./langgraph_search_router_agent.ipynb) 에 있습니다.

```python
workflow.add_conditional_edges(
    'action',              # 1단계 : 어디서 출발할지 (시작 노드)
    should_continue,       # 2단계 : 어떤 기준으로 판단할지 (라우팅 함수 — 상태를 읽고 '이름' 을 돌려줌)
    {'continue': 'action', # 3단계 : 그 이름이면 어디로 보낼지 (경로 매핑)
     'end': END},
)
```

- 원본에서 고친 부분 · 보충은 **📝** 로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 이 노트는 LLM 을 부르지 않는 **모의 노드**만 써서 **API 키 없이 모든 셀을 실제로 실행**했다.

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langgraph` | `StateGraph` · `START` · `END` · `add_conditional_edges` · `GraphRecursionError` | `pip install langgraph` |

**API 키 · 인터넷 필요 없음** (정리 환경 : `langgraph 1.2.21` · `langchain-core 1.8.0` · `pydantic 2.14.3`). 맨 위부터 **모두 실행**. 그래프 그림은 `draw_mermaid()` 글자로 출력한다 — Mermaid 를 지원하는 편집기 · GitHub 에 붙이면 그림이 된다. **점선(`-.->`) = 조건부 엣지**, 실선(`-->`) = 일반 엣지.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph

In [2]:
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.errors import GraphRecursionError

def show(app):
    # Mermaid 글자에서 화살표 줄만 (점선 -.-> = 조건부 엣지)
    for line in app.get_graph().draw_mermaid().splitlines():
        if ('--' in line or '-.' in line) and not line.startswith('---'):
            print(line.strip())

## 1. 왜 조건부 엣지인가 (교재 p.96~98)

| | 지난 시간 (일반 엣지 `add_edge`) | 오늘 (조건부 엣지 `add_conditional_edges`) |
|---|---|---|
| 다음 노드 | 그래프를 만들 때 **하나로 고정** | 실행할 때 **상태를 보고** 고른다 |
| 비유 | 컨베이어 벨트 — 중간에 문제가 생겨도 다음 칸으로 | 갈림길 표지판 — A 다음에 B · C · D 중 하나 |
| 할 수 있는 것 | A → B → C | 분기(검색이 필요할 때만 검색) · **되돌아가기**(결과가 부족하면 다시) · 끝내기(`END`) |

## 2. 3단계 매핑 — 교재 p.99 코드를 그대로 실행

교재의 `should_continue` 예시를 '숫자를 1씩 올리다가 3 이 되면 끝' 으로 채워 실행했다. `'continue': 'action'` 은 **자기 자신으로 되돌아가는 엣지**다.

In [3]:
class CountState(TypedDict):
    n: int

def action(state: CountState) -> dict:
    print(f"  action 노드 : n = {state['n']} → {state['n'] + 1}")
    return {'n': state['n'] + 1}

def should_continue(state: CountState) -> str:   # 2단계 : 상태를 읽고 '이름' 만 돌려준다
    decision = 'continue' if state['n'] < 3 else 'end'
    print(f"  should_continue : n = {state['n']} → '{decision}'")
    return decision

workflow = StateGraph(CountState)
workflow.add_node('action', action)
workflow.add_edge(START, 'action')
workflow.add_conditional_edges(
    'action',                                    # 1단계 : 시작 노드
    should_continue,                             # 2단계 : 조건 판단 함수
    {'continue': 'action', 'end': END},          # 3단계 : 이름 → 노드
)
loop_app = workflow.compile()
print('결과 :', loop_app.invoke({'n': 0}))
show(loop_app)

  action 노드 : n = 0 → 1
  should_continue : n = 1 → 'continue'
  action 노드 : n = 1 → 2
  should_continue : n = 2 → 'continue'
  action 노드 : n = 2 → 3
  should_continue : n = 3 → 'end'
결과 : {'n': 3}
__start__ --> action;
action -. &nbsp;end&nbsp; .-> __end__;
action -. &nbsp;continue&nbsp; .-> action;


- 라우팅 함수는 **노드가 아니다** — 상태를 바꾸지 못하고(돌려준 값은 상태에 들어가지 않는다) '다음에 갈 곳의 이름' 만 정한다
- 매핑의 키(`'continue'`)와 노드 이름(`'action'`)이 다르면 Mermaid 그림의 점선 위에 키가 글자로 붙는다

## 3. 매핑을 쓰는 세 가지 방법 — 그림이 달라진다

실습(p.101)과 같은 모양의 그래프로 비교했다. 라우팅 함수가 **노드 이름을 바로** 돌려주는 경우다.

In [4]:
class S(TypedDict):
    needs_search: bool

def classifier(state: S) -> dict:  return {}
def search(state: S) -> dict:      return {}
def answer(state: S) -> dict:      return {}

def route_plain(state: S):                                          # (나) 타입 힌트 없음
    return 'search_node' if state['needs_search'] else 'answer_node'

def route_literal(state: S) -> Literal['search_node', 'answer_node']:   # (다) Literal 로 갈 곳을 적어 둠
    return 'search_node' if state['needs_search'] else 'answer_node'

def build(router, mapping=None):
    g = StateGraph(S)
    g.add_node('classifier_node', classifier); g.add_node('search_node', search); g.add_node('answer_node', answer)
    g.add_edge(START, 'classifier_node')
    if mapping:
        g.add_conditional_edges('classifier_node', router, mapping)
    else:
        g.add_conditional_edges('classifier_node', router)
    g.add_edge('search_node', 'answer_node'); g.add_edge('answer_node', END)
    return g.compile()

ways = {
    '(가) 매핑 dict (교재)':  build(route_plain, {'search_node': 'search_node', 'answer_node': 'answer_node'}),
    '(나) 매핑 없음 · 힌트 없음': build(route_plain),
    '(다) 매핑 없음 · Literal':  build(route_literal),
}
for name, app in ways.items():
    path = [list(step)[0] for step in app.stream({'needs_search': True})]
    print(f'■ {name}  — 실행 경로 : {path}')
    show(app); print()

■ (가) 매핑 dict (교재)  — 실행 경로 : ['classifier_node', 'search_node', 'answer_node']
__start__ --> classifier_node;
classifier_node -.-> answer_node;
classifier_node -.-> search_node;
search_node --> answer_node;
answer_node --> __end__;

■ (나) 매핑 없음 · 힌트 없음  — 실행 경로 : ['classifier_node', 'search_node', 'answer_node']
__start__ --> classifier_node;
classifier_node --> __end__;

■ (다) 매핑 없음 · Literal  — 실행 경로 : ['classifier_node', 'search_node', 'answer_node']
__start__ --> classifier_node;
classifier_node -.-> answer_node;
classifier_node -.-> search_node;
search_node --> answer_node;
answer_node --> __end__;



| 방법 | 실행 | 그래프 그림 |
|---|---|---|
| (가) 매핑 dict | 정상 | 갈 수 있는 곳 2개가 점선 |
| (나) 매핑 없음 · 힌트 없음 | **정상** (`search_node` → `answer_node`) | **틀린 그림** — `classifier_node --> __end__` 로 그려진다 (어디로 갈지 LangGraph 가 미리 알 수 없어서) |
| (다) `-> Literal[...]` | 정상 | (가)와 같다 |

- 📝 실행만 보면 셋 다 같지만, **그림으로 설계를 확인하려면 매핑 dict 나 `Literal` 을 써야 한다**

## 4. 라우팅 함수가 없는 이름을 돌려주면

In [5]:
def route_typo(state: S):
    return 'searchnode'          # 밑줄(_) 하나 빠진 오타

# (가) 매핑 dict 가 있으면 → 바로 오류
try:
    build(route_typo, {'search_node': 'search_node', 'answer_node': 'answer_node'}).invoke({'needs_search': True})
except Exception as e:
    print('(가) 매핑 있음 :', type(e).__name__, e)

(가) 매핑 있음 : KeyError 'searchnode'


In [6]:
# (나) 매핑이 없으면 → 경고 한 줄만 남기고 그래프가 '조용히' 끝난다 (answer_node 를 지나지 않음)
path = [list(step)[0] for step in build(route_typo).stream({'needs_search': True})]
print('(나) 매핑 없음 · 실행 경로 :', path)

Task classifier_node with path ('__pregel_pull', 'classifier_node') wrote to unknown channel branch:to:searchnode, ignoring it.


(나) 매핑 없음 · 실행 경로 : ['classifier_node']


- 매핑이 없으면 오타가 **오류 없이** 그래프를 멈춘다(`wrote to unknown channel … ignoring it` 경고만). 답변 노드를 건너뛴 채 끝나므로 찾기 어려운 버그가 된다 → **매핑 dict 를 써서 `KeyError` 로 드러나게** 하자
- 라우팅 함수가 `END` 를 돌려주면 그 자리에서 끝난다 (`{'end': END}` 처럼 매핑에 넣어도 된다)

## 5. 되돌아가는 엣지와 `recursion_limit`

조건이 끝내 `'end'` 를 주지 않으면 무한 반복이다. LangGraph 는 **한 번 실행에서 거칠 수 있는 단계 수**(`recursion_limit`)를 넘으면 멈춘다.

In [7]:
def never_end(state: CountState) -> str:
    return 'continue'                         # 끝나는 조건이 없는 라우팅 함수

g = StateGraph(CountState)
g.add_node('action', lambda s: {'n': s['n'] + 1})
g.add_edge(START, 'action')
g.add_conditional_edges('action', never_end, {'continue': 'action', 'end': END})
forever = g.compile()

try:
    forever.invoke({'n': 0}, {'recursion_limit': 5})          # 단계 수를 5 로 제한
except GraphRecursionError as e:
    print('limit=5 :', type(e).__name__, '—', str(e).splitlines()[0])

try:
    forever.invoke({'n': 0})                                 # 기본값
except GraphRecursionError as e:
    print('기본값  :', type(e).__name__, '—', str(e).splitlines()[0])

limit=5 : GraphRecursionError — Recursion limit of 5 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


기본값  : GraphRecursionError — Recursion limit of 10007 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


- 📝 많은 자료에 '기본 `recursion_limit` = 25' 라고 적혀 있지만, **정리 환경(langgraph 1.2.21)의 기본값은 10007** 이었다(위 출력 · 소스 `langgraph/_internal/_config.py` 에서 환경 변수 `LANGGRAPH_DEFAULT_RECURSION_LIMIT` 이 없으면 10007). 25 는 `langchain_core` 의 기본값이다. 25 에서 멈춰 주리라 믿으면 안 된다 — LLM 노드가 들어간 반복이면 **수천 번 API 를 부를 수 있다**
- 되돌아가는 그래프를 만들면 ① 상태에 **횟수**(`retries`)를 두고 라우팅 함수가 횟수로도 끝내게 하고 ② `invoke(…, {'recursion_limit': N})` 으로 안전장치를 건다

## 정리

| 할 일 | 코드 |
|---|---|
| 조건부 엣지 | `workflow.add_conditional_edges('시작 노드', 라우팅함수, {'이름': '노드', 'end': END})` |
| 라우팅 함수 | `def route(state) -> Literal['a', 'b']:` — 상태를 읽고 **이름만** 돌려줌 (상태는 못 바꿈) |
| 시작점 | `workflow.set_entry_point('노드')` = `workflow.add_edge(START, '노드')` |
| 그림 확인 | `app.get_graph().draw_mermaid()` — 점선 = 조건부 엣지 · 매핑 / `Literal` 이 없으면 그림이 틀린다 |
| 없는 이름 | 매핑 있음 → `KeyError` / 매핑 없음 → 경고만 남기고 끝남 |
| 반복 안전장치 | 상태에 횟수 + `invoke(…, {'recursion_limit': N})` → 넘으면 `GraphRecursionError` |